# LE8 - Link Prediction in Football Networks

This notebook applies link prediction algorithms to football networks to identify potential future connections:

1. **Player Pass Networks**: Which players might start passing to each other
2. **Team Match Networks**: Which teams might play against each other

**1. Link Prediction** predicts which edges are likely to form in a network based on current network structure and patterns.

### Applications in Football Networks

**Pass Networks (Players)**:
- Identify which players might develop new passing connections
- Discover potential tactical partnerships
- Players who share many passing targets but don't pass directly to each other

**Match Networks (Teams)**:
- Predict which teams might face each other in future tournaments
- Based on existing match patterns
- Teams with common opponents but haven't played each other


### Key Concept: Neighborhood Function N(·)

Throughout all algorithms, we use $N(v)$ to denote the set of neighbors of node $v$ in the network.

All algorithms are based on neighborhood information and structural similarities between nodes.

Alli link predictions in python library networkx vorhande

In [1]:
# Import required libraries
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict, Counter
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.common import load_json_fixed, setup_logger
from src.config import (
    DATA_DIR,
    PASS_NETWORKS_DIR,
    MATCH_NETWORKS_DIR,
    PLAYER_TEAM_NETWORKS_DIR,
)

# Setup plotting
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

print("Imports successful!")
print(f"NetworkX version: {nx.__version__}")

Imports successful!
NetworkX version: 3.6.1


In [2]:
# Load networks
print("=== Loading Networks ===\n")

# Pass Network (directed, weighted)
pass_network_path = PASS_NETWORKS_DIR / 'pass_network_Germany.gexf'
G_pass_dir = nx.read_gexf(pass_network_path)
print(f"Pass Network (Germany): {G_pass_dir.number_of_nodes()} nodes, {G_pass_dir.number_of_edges()} edges")

# Convert to undirected for some centrality measures
G_pass = G_pass_dir.to_undirected()

# Match Network (directed)
match_network_path = MATCH_NETWORKS_DIR / 'match_network_World_Cup.gexf'
G_match_dir = nx.read_gexf(match_network_path)
print(f"Match Network (World Cup): {G_match_dir.number_of_nodes()} nodes, {G_match_dir.number_of_edges()} edges")

# Convert to undirected
G_match = G_match_dir.to_undirected()

# Unified Pass Network
unified_pass_path = PASS_NETWORKS_DIR / 'pass_network_unified.gexf'
if unified_pass_path.exists():
    G_unified_dir = nx.read_gexf(unified_pass_path)
    G_unified = G_unified_dir.to_undirected()
    print(f"Unified Pass Network: {G_unified.number_of_nodes()} nodes, {G_unified.number_of_edges()} edges")
else:
    G_unified = None
    print("Unified pass network not found")

print("\nNetworks loaded successfully!")

=== Loading Networks ===

Pass Network (Germany): 473 nodes, 8786 edges
Match Network (World Cup): 32 nodes, 126 edges
Unified Pass Network: 3028 nodes, 33178 edges

Networks loaded successfully!


## 2. Common Neighbors

**Common Neighbors** counts how many mutual neighbors two nodes share. It's the simplest link prediction metric.

### Intuition

If two players/teams have many common neighbors, they're likely to form a connection. This is based on:
- **Triadic closure**: "friend of a friend becomes a friend"
- In football: players who pass to the same teammates may eventually pass directly to each other

### Mathematical Definition

For two nodes $x$ and $y$ that are not directly connected:

$$\text{common\_neighbors}(x, y) = |N(x) \cap N(y)|$$

where:
- $N(x)$ = set of neighbors of node $x$
- $\cap$ = set intersection
- $|\cdot|$ = cardinality (size of set)

### Example

If player A passes to players {B, D, E} and player C passes to players {B, D, F}:
- Common neighbors: $\{B, D\}$
- Score: $|{B, D}| = 2$

### Interpretation

- **Higher score** → more likely candidate for a new edge
- Simple but effective baseline
- Works well in dense, clustered networks
- **Limitation**: Doesn't account for how many total neighbors each node has

## 3. Jaccard Coefficient

**Jaccard Coefficient** normalizes common neighbors by the total number of neighbors both nodes have.

### Intuition

Having 5 common neighbors is more meaningful when both nodes have only 10 neighbors total, versus when they have 100 neighbors each. Jaccard accounts for this relative overlap.

### Mathematical Definition

$$\text{jaccard\_coefficient}(x, y) = \frac{|N(x) \cap N(y)|}{|N(x) \cup N(y)|}$$

where:
- Numerator: number of common neighbors (intersection)
- Denominator: total unique neighbors of both nodes (union)
- $\cup$ = set union

### Properties

- **Range**: $[0, 1]$
- **Value = 1**: All neighbors are shared
- **Value = 0**: No common neighbors

### Example

Player A neighbors: $\{B, D, E\}$, Player C neighbors: $\{B, D, F\}$

$$\text{jaccard\_coefficient}(A, C) = \frac{|\{B, D\}|}{|\{B, D, E, F\}|} = \frac{2}{4} = 0.5$$

### Interpretation

- **Advantage over Common Neighbors**: Controls for highly connected nodes
- Better when node degrees vary significantly
- Normalized score allows fair comparison across different node pairs

## 4. Adamic-Adar Index

**Adamic-Adar** weights common neighbors inversely by their degree, giving more importance to rare shared connections.

### Intuition

A common neighbor with few connections (rare connection) is more significant than a highly connected common neighbor (hub).

**Football example**: Sharing a connection through a specialized defensive midfielder is more meaningful than sharing a connection through a central playmaker who passes to everyone.

### Mathematical Definition

$$\text{adamic\_adar}(x, y) = \sum_{u \in N(x) \cap N(y)} \frac{1}{\log(|N(u)|)}$$

where:
- Sum over all common neighbors $u$
- Each neighbor weighted by $\frac{1}{\log(\text{degree of } u)}$
- Logarithm softens the penalty for high-degree nodes

### Why Logarithm?

- Reduces the penalty compared to direct inverse of degree
- Balance between treating all common neighbors equally vs. heavily penalizing hubs
- Empirically performs well across many network types

### Example

Player A and C share neighbors B and D, both with degree 3:

$$\text{adamic\_adar}(A, C) = \frac{1}{\ln(3)} + \frac{1}{\ln(3)} = \frac{2}{\ln(3)} \approx 1.82$$

### Interpretation

- **Higher score**: More valuable (rare) shared connections
- Often ranks predictions differently than Common Neighbors
- Effective in networks with hub structures

## 5. Resource Allocation

**Resource Allocation** models resources flowing from one node to another through common neighbors.

### Intuition

Imagine node $x$ sending resources to node $y$ through intermediate neighbors. Each common neighbor distributes resources equally among all its neighbors. Neighbors with many connections dilute the resource flow.

**Football analogy**: Tactical information flows through common teammates, but players with many connections spread their influence thin.

### Mathematical Definition

$$\text{resource\_allocation}(x, y) = \sum_{u \in N(x) \cap N(y)} \frac{1}{|N(u)|}$$

where:
- Each common neighbor $u$ contributes $\frac{1}{\text{degree of } u}$
- No logarithm (stricter penalty than Adamic-Adar)

### Comparison with Adamic-Adar

- **Resource Allocation**: $\frac{1}{k}$ penalty
- **Adamic-Adar**: $\frac{1}{\log(k)}$ penalty
- Resource Allocation penalizes high-degree common neighbors more heavily

### Example

Player A and C share neighbors B (degree 3) and D (degree 3):

$$\text{resource\_allocation}(A, C) = \frac{1}{3} + \frac{1}{3} = \frac{2}{3} \approx 0.67$$

### Interpretation

- Emphasizes rare connections even more than Adamic-Adar
- May give very different rankings than simpler methods
- Can identify non-obvious potential connections

## 6. Preferential Attachment

**Preferential Attachment** is based on the "rich get richer" phenomenon: popular nodes attract more connections.

### Intuition

Nodes with many existing connections are more likely to form new connections, regardless of network structure or common neighbors.

**Football examples**:
- Star players (high degree) naturally develop more passing connections
- Popular teams play more matches across competitions

### Mathematical Definition

$$\text{preferential\_attachment}(x, y) = |N(x)| \times |N(y)|$$

Simply the product of the degrees of both nodes.

### Theoretical Background

Based on the **Barabási-Albert model** of scale-free networks, where connection probability is proportional to node degree.

### Example

Player A has 3 neighbors, Player C has 3 neighbors:

$$\text{preferential\_attachment}(A, C) = 3 \times 3 = 9$$

Player A has 3 neighbors, Player G has 4 neighbors:

$$\text{preferential\_attachment}(A, G) = 3 \times 4 = 12$$

### Interpretation

- **Fundamentally different**: Does NOT consider common neighbors at all
- Only considers node popularity (degree)
- May predict connections between unrelated but popular nodes
- Captures different aspect of link formation than neighborhood-based methods

## 7. Comparing Algorithms

### Different Perspectives on Link Formation

Each algorithm captures different mechanisms:

1. **Common Neighbors**: Triadic closure, local clustering
2. **Jaccard Coefficient**: Relative similarity in neighborhoods
3. **Adamic-Adar**: Valuable rare connections
4. **Resource Allocation**: Information/resource flow
5. **Preferential Attachment**: Popularity and growth dynamics

### Expected Patterns in Football Networks

**Pass Networks**:
- High clustering → neighborhood-based methods should identify meaningful patterns
- Star players (hubs) → Preferential Attachment might rank them highly
- Tactical units → Adamic-Adar/Resource Allocation might find specialist connections

**Match Networks**:
- Tournament structures → Common Neighbors might group competitive levels
- Popular teams → Preferential Attachment relevant
- Regional patterns → May not be captured by purely structural methods
